### ДЗ_3 Линейная классификация

* Измените функцию calc_logloss так, чтобы нули по возможности не попадали в np.log.
* Подберите аргументы функции eval_model для логистической регрессии таким образом, чтобы log loss был минимальным.
* Создайте функцию calc_pred_proba, возвращающую предсказанную вероятность класса 1 (на вход подаются W, который уже посчитан функцией eval_model и X, на выходе - массив y_pred_proba).
* Создайте функцию calc_pred, возвращающую предсказанный класс (на вход подаются W, который уже посчитан функцией eval_model и X, на выходе - массив y_pred).
* Посчитайте Accuracy, матрицу ошибок, точность и полноту, а также F1 score.
* Могла ли модель переобучиться? Почему?

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

In [3]:
# Датасет по обучению абитуриентов (теги: стаж преподавателя, стоимость 1 часа, рейтинг. таргет: поступление в ВУЗ)
X = np.array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
              [1, 1, 2, 1, 3, 0, 5, 10, 1, 2],
              [500, 700, 750, 600, 1450,
               800, 1500, 2000, 450, 1000],
              [1, 1, 2, 1, 2, 1, 3, 3, 1, 2]], dtype=np.float64)

y = np.array([0, 0, 1, 0, 1, 0, 1, 0, 1, 1], dtype=np.float64)

In [4]:
def calc_std_feat(x):
    """Нормализация признака"""
    res = (x - x.mean()) / x.std()
    return res

In [5]:
def sigmoid(z):
    """Сигмоида с защитой от переполнения"""
    # Защита от переполнения: для больших отрицательных z используем альтернативную формулу
    z = np.clip(z, -500, 500)  # Ограничиваем значения для предотвращения overflow
    res = 1 / (1 + np.exp(-z))
    return res


In [6]:
# Задание 1: Измените функцию calc_logloss так, чтобы нули по возможности не попадали в np.log
def calc_logloss(y, y_pred, epsilon=1e-15):
    """
    Вычисление log loss с защитой от нулей в логарифме
    
    Параметры:
    ----------
    y : array-like
        Истинные метки (0 или 1)
    y_pred : array-like
        Предсказанные вероятности
    epsilon : float
        Малое значение для предотвращения log(0)
    
    Возвращает:
    -----------
    float
        Значение log loss
    """
    # Ограничиваем y_pred в диапазоне [epsilon, 1-epsilon]
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    err = -np.mean(y * np.log(y_pred) + (1.0 - y) * np.log(1.0 - y_pred))
    return err

In [7]:
# Задание 2: Подберите аргументы функции eval_model для минимизации log loss
def eval_model(X, y, iterations, alpha=1e-4, verbose=False):
    """
    Обучение модели логистической регрессии методом градиентного спуска
    
    Параметры:
    ----------
    X : array-like
        Матрица признаков (shape: [n_features, n_samples])
    y : array-like
        Вектор меток
    iterations : int
        Количество итераций
    alpha : float
        Скорость обучения (learning rate)
    verbose : bool
        Выводить ли информацию о процессе обучения
    
    Возвращает:
    -----------
    array
        Вектор весов W
    """
    np.random.seed(42)
    W = np.random.randn(X.shape[0])
    n = X.shape[1]
    
    for i in range(1, iterations + 1):
        z = np.dot(W, X)
        y_pred = sigmoid(z)
        err = calc_logloss(y, y_pred)
        
        # Градиентный спуск
        W -= alpha * (1/n * np.dot((y_pred - y), X.T))
        
        # Вывод информации каждые 10% итераций
        if verbose and i % (iterations // 10) == 0:
            print(f"Iteration {i}: Log Loss = {err:.6f}, W = {W}")
    
    return W

In [8]:
# Задание 3: Создайте функцию calc_pred_proba
def calc_pred_proba(W, X):
    """
    Вычисление предсказанных вероятностей класса 1
    
    Параметры:
    ----------
    W : array-like
        Вектор весов модели
    X : array-like
        Матрица признаков (shape: [n_features, n_samples])
    
    Возвращает:
    -----------
    array
        Предсказанные вероятности класса 1
    """
    z = np.dot(W, X)
    y_pred_proba = sigmoid(z)
    return y_pred_proba

In [9]:
# Задание 4: Создайте функцию calc_pred
def calc_pred(W, X, threshold=0.5):
    """
    Вычисление предсказанных классов
    
    Параметры:
    ----------
    W : array-like
        Вектор весов модели
    X : array-like
        Матрица признаков (shape: [n_features, n_samples])
    threshold : float
        Порог для классификации (по умолчанию 0.5)
    
    Возвращает:
    -----------
    array
        Предсказанные классы (0 или 1)
    """
    y_pred_proba = calc_pred_proba(W, X)
    y_pred = (y_pred_proba >= threshold).astype(int)
    return y_pred

In [10]:
# Подбор оптимальных параметров для минимизации log loss
print("=" * 60)
print("Задание 2: Подбор оптимальных параметров")
print("=" * 60)

Задание 2: Подбор оптимальных параметров


In [11]:
# Тестируем различные комбинации параметров
best_loss = float('inf')
best_params = None
best_W = None

In [12]:
# Пробуем разные значения learning rate и количества итераций
alphas = [1e-5, 1e-4, 1e-3, 5e-3]
iterations_list = [1000, 5000, 10000, 20000]

In [13]:
print("\nПоиск оптимальных параметров...")
for alpha in alphas:
    for iterations in iterations_list:
        W = eval_model(X, y, iterations=iterations, alpha=alpha, verbose=False)
        y_pred_proba = calc_pred_proba(W, X)
        loss = calc_logloss(y, y_pred_proba)
        
        if loss < best_loss:
            best_loss = loss
            best_params = (iterations, alpha)
            best_W = W.copy()
        
        print(f"alpha={alpha:.0e}, iterations={iterations}: log_loss={loss:.6f}")

print(f"\nЛучшие параметры: iterations={best_params[0]}, alpha={best_params[1]:.0e}")
print(f"Минимальный log loss: {best_loss:.6f}")


Поиск оптимальных параметров...
alpha=1e-05, iterations=1000: log_loss=0.909047
alpha=1e-05, iterations=5000: log_loss=0.908360
alpha=1e-05, iterations=10000: log_loss=0.907502
alpha=1e-05, iterations=20000: log_loss=0.905786
alpha=1e-04, iterations=1000: log_loss=14.810559
alpha=1e-04, iterations=5000: log_loss=14.184215
alpha=1e-04, iterations=10000: log_loss=14.510747
alpha=1e-04, iterations=20000: log_loss=14.436854
alpha=1e-03, iterations=1000: log_loss=10.937941
alpha=1e-03, iterations=5000: log_loss=17.269388
alpha=1e-03, iterations=10000: log_loss=17.269388
alpha=1e-03, iterations=20000: log_loss=17.269388
alpha=5e-03, iterations=1000: log_loss=17.269788
alpha=5e-03, iterations=5000: log_loss=17.269388
alpha=5e-03, iterations=10000: log_loss=17.269388
alpha=5e-03, iterations=20000: log_loss=17.269788

Лучшие параметры: iterations=20000, alpha=1e-05
Минимальный log loss: 0.905786


In [14]:
# Используем лучшие параметры
W = best_W

In [21]:
# Задание 5: Посчитайте метрики

y_pred_proba = calc_pred_proba(W, X)
y_pred = calc_pred(W, X)

In [22]:
# Accuracy
accuracy = accuracy_score(y, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")


Accuracy: 0.7000


In [17]:
# Матрица ошибок (Confusion Matrix)
cm = confusion_matrix(y, y_pred)
print(f"\nМатрица ошибок (Confusion Matrix):")
print("                Predicted")
print("                0    1")
print(f"Actual  0    {cm[0,0]:4d} {cm[0,1]:4d}")
print(f"        1    {cm[1,0]:4d} {cm[1,1]:4d}")


Матрица ошибок (Confusion Matrix):
                Predicted
                0    1
Actual  0       5    0
        1       3    2


In [18]:
# Точность (Precision)
precision = precision_score(y, y_pred, zero_division=0)
print(f"\nТочность (Precision): {precision:.4f}")

# Полнота (Recall)
recall = recall_score(y, y_pred, zero_division=0)
print(f"Полнота (Recall): {recall:.4f}")

# F1 score
f1 = f1_score(y, y_pred, zero_division=0)
print(f"F1 score: {f1:.4f}")


Точность (Precision): 1.0000
Полнота (Recall): 0.4000
F1 score: 0.5714


In [19]:
# Дополнительная информация
print(f"\nДополнительная информация:")
print(f"Предсказанные вероятности: {y_pred_proba}")
print(f"Предсказанные классы: {y_pred}")
print(f"Истинные классы: {y}")


Дополнительная информация:
Предсказанные вероятности: [0.4510807  0.26412064 0.53955422 0.3519491  0.0528257  0.21533753
 0.1359824  0.00948128 0.50269587 0.29382149]
Предсказанные классы: [0 0 1 0 0 0 0 0 1 0]
Истинные классы: [0. 0. 1. 0. 1. 0. 1. 0. 1. 1.]


In [20]:
# Задание 6: Могла ли модель переобучиться? Почему?
print("\n" + "=" * 60)
print("Задание 6: Анализ переобучения")
print("=" * 60)
print("""
Модель НЕ могла переобучиться по следующим причинам:

1. МАЛЫЙ РАЗМЕР ДАТАСЕТА:
   - У нас всего 10 образцов в обучающей выборке
   - Это слишком мало для переобучения, так как модель не имеет достаточно данных
   для запоминания сложных паттернов

2. ПРОСТАЯ МОДЕЛЬ:
   - Логистическая регрессия - это линейная модель с небольшим количеством параметров
   - У нас всего 4 признака, что означает 4 веса + bias (всего 5 параметров)
   - При 10 образцах и 5 параметрах модель не имеет избыточной сложности

3. ОТСУТСТВИЕ РАЗДЕЛЕНИЯ НА TRAIN/TEST:
   - Мы обучаем и тестируем на одних и тех же данных
   - Но даже в этом случае, из-за простоты модели и малого размера данных,
   переобучение маловероятно

4. РЕГУЛЯРИЗАЦИЯ:
   - В нашей реализации нет явной регуляризации, но малый размер данных
   и простая модель естественным образом ограничивают сложность

ВЫВОД: Модель скорее НЕДООБУЧЕНА из-за малого количества данных,
а не переобучена. Для более точной оценки нужна валидационная выборка
или кросс-валидация.
""")



Задание 6: Анализ переобучения

Модель НЕ могла переобучиться по следующим причинам:

1. МАЛЫЙ РАЗМЕР ДАТАСЕТА:
   - У нас всего 10 образцов в обучающей выборке
   - Это слишком мало для переобучения, так как модель не имеет достаточно данных
   для запоминания сложных паттернов

2. ПРОСТАЯ МОДЕЛЬ:
   - Логистическая регрессия - это линейная модель с небольшим количеством параметров
   - У нас всего 4 признака, что означает 4 веса + bias (всего 5 параметров)
   - При 10 образцах и 5 параметрах модель не имеет избыточной сложности

3. ОТСУТСТВИЕ РАЗДЕЛЕНИЯ НА TRAIN/TEST:
   - Мы обучаем и тестируем на одних и тех же данных
   - Но даже в этом случае, из-за простоты модели и малого размера данных,
   переобучение маловероятно

4. РЕГУЛЯРИЗАЦИЯ:
   - В нашей реализации нет явной регуляризации, но малый размер данных
   и простая модель естественным образом ограничивают сложность

ВЫВОД: Модель скорее НЕДООБУЧЕНА из-за малого количества данных,
а не переобучена. Для более точной оценки ну